# AI-Powered Bird Call Detection — ML Extension
### Complete pipeline: data → augmentation → two-phase training → K-fold CV → evaluation → ONNX export


In [ ]:
# ── 0. Setup paths & imports ──────────────────────────────────────────
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent  # adjust if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings; warnings.filterwarnings('ignore')
import yaml, json, time, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s')
logger = logging.getLogger('notebook')
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── 1. Load config ────────────────────────────────────────────────────
with open(PROJECT_ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

# Ensure directories exist
for key in ['raw_audio_dir','processed_dir','cache_dir','models_dir','logs_dir','results_dir','figures_dir']:
    Path(config['paths'][key]).mkdir(parents=True, exist_ok=True)

NUM_CLASSES = config['model']['num_classes']
CLASS_NAMES = ([s['common_name'] for s in config['species']['target_species']] +
               [s['common_name'] for s in config['species']['background_species']])
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

In [ ]:
# ── 2. Data collection via Xeno-Canto ────────────────────────────────
from src.data.xeno_canto_api import XenoCantoAPI
from src.data.dataset import build_multiclass_dataframe

api = XenoCantoAPI(config)

# collect target species
target_recordings = []
for sp in config['species']['target_species']:
    query = f'sp:"{sp["scientific_name"]}" q:A'
    recs = api.search_recordings(query, max_results=config['xeno_canto']['recordings_per_species'])
    for r in recs:
        r['target_species'] = sp['scientific_name']
        r['common_name']    = sp['common_name']
        r['class_id']       = sp['class_id']
    target_recordings.extend(recs)
    print(f"  {sp['common_name']}: {len(recs)} recordings")

# collect background species
bg_recordings = []
bg_per = max(50, len(target_recordings) // len(config['species']['background_species']))
for sp in config['species']['background_species']:
    query = f'sp:"{sp["scientific_name"]}" q:A'
    recs = api.search_recordings(query, max_results=bg_per)
    for r in recs:
        r['target_species'] = sp['scientific_name']
        r['common_name']    = sp['common_name']
        r['class_id']       = sp['class_id']
    bg_recordings.extend(recs)
    print(f"  {sp['common_name']} (bg): {len(recs)} recordings")

all_recs = target_recordings + bg_recordings
raw_df = pd.DataFrame(all_recs)
full_df = build_multiclass_dataframe(
    raw_df, config['species']['target_species'], config['species']['background_species'])
print(f'\nTotal recordings: {len(full_df)}')
print(full_df['label'].value_counts().to_string())

In [ ]:
# ── 3. Download audio files ───────────────────────────────────────────
download_dir = Path(config['paths']['raw_audio_dir'])
downloaded_df = api.batch_download_audio(full_df, download_dir, max_workers=3)
# keep only successfully downloaded
downloaded_df = downloaded_df[downloaded_df['download_success'] == True].copy()
downloaded_df = downloaded_df[downloaded_df['local_path'].notna()].copy()
print(f'Downloaded: {len(downloaded_df)} files')
print(downloaded_df['label'].value_counts().to_string())

In [ ]:
# ── 4. Audio processor + sample visualisation ────────────────────────
from src.data.audio_processor import AudioProcessor

audio_proc = AudioProcessor(config)

# visualise one sample per class
sample_paths = (
    downloaded_df.groupby('label')
    .first()
    .reset_index()[['label','local_path','common_name']]
)

fig, axes = plt.subplots(2, min(4, len(sample_paths)), figsize=(16, 6))
for i, (_, row) in enumerate(sample_paths.iterrows()):
    if i >= 4: break
    ax_wave, ax_mel = axes[0, i], axes[1, i]
    audio = audio_proc.load_audio(row['local_path'])
    mel   = audio_proc.compute_mel_spectrogram(audio)
    ax_wave.plot(audio, linewidth=0.3)
    ax_wave.set_title(row['common_name'], fontsize=8)
    ax_wave.set_ylabel('Amplitude')
    ax_mel.imshow(mel, origin='lower', aspect='auto', cmap='magma')
    ax_mel.set_ylabel('Mel bin')
    ax_mel.set_xlabel('Frame')
plt.suptitle('Sample waveforms and mel-spectrograms')
plt.tight_layout()
plt.savefig(Path(config['paths']['figures_dir']) / 'sample_spectrograms.png', dpi=120)
plt.show()

In [ ]:
# ── 5. Class weights (for Focal Loss) ────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

labels_array = downloaded_df['label'].values
unique_classes = np.sort(np.unique(labels_array))
cw = compute_class_weight('balanced', classes=unique_classes, y=labels_array)
class_weights = torch.FloatTensor(cw)
print('Class weights:', {CLASS_NAMES[c]: round(float(w), 3)
                         for c, w in zip(unique_classes, cw)})

In [ ]:
# ── 6. Build model ────────────────────────────────────────────────────
from src.models.efficientnet_classifier import create_model, count_parameters

model = create_model(config)
total, trainable = count_parameters(model)
print(f'Total params: {total:,} | Trainable: {trainable:,}')

In [ ]:
# ── 7a. OPTION A — K-Fold cross-validation (recommended) ─────────────
# Set USE_KFOLD = False to do a single train/val/test split instead.
USE_KFOLD = config['training'].get('use_kfold', True)

from src.training.trainer import BirdDetectionTrainer

trainer = BirdDetectionTrainer(model, config, audio_proc)

if USE_KFOLD:
    fold_results = trainer.kfold_train(downloaded_df, class_weights=class_weights)
    f1s = [r['best_val_f1'] for r in fold_results]
    print(f'\nK-Fold F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')

In [ ]:
# ── 7b. OPTION B — Single train/val/test split ───────────────────────
if not USE_KFOLD:
    from sklearn.model_selection import train_test_split
    tc = config['training']

    train_df, temp_df = train_test_split(
        downloaded_df, test_size=tc['validation_split'] + tc['test_split'],
        stratify=downloaded_df['label'], random_state=tc['random_seed'])
    val_df, test_df = train_test_split(
        temp_df,
        test_size=tc['test_split'] / (tc['validation_split'] + tc['test_split']),
        stratify=temp_df['label'], random_state=tc['random_seed'])

    print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
    single_result = trainer.train_two_phase(train_df, val_df,
                                            class_weights=class_weights, fold=0)

In [ ]:
# ── 8. Full evaluation on held-out test set ──────────────────────────
from src.data.dataset import BirdCallDataset
from torch.utils.data import DataLoader
from src.evaluation.evaluator import ModelEvaluator

# If K-fold, use the last fold's val as test (or provide a separate test set)
if USE_KFOLD:
    from sklearn.model_selection import train_test_split as tts
    _, test_df = tts(downloaded_df, test_size=0.15,
                     stratify=downloaded_df['label'],
                     random_state=config['training']['random_seed'])

test_ds = BirdCallDataset(test_df, audio_proc, validate_files=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         num_workers=2, pin_memory=torch.cuda.is_available())

evaluator = ModelEvaluator(trainer.model, config, CLASS_NAMES, device=trainer.device)
metrics = evaluator.evaluate(test_loader)

print(f"\n{'='*50}")
print(f"  Accuracy        : {metrics['accuracy']:.4f}")
print(f"  F1 (weighted)   : {metrics['f1_weighted']:.4f}")
print(f"  Precision       : {metrics['precision_weighted']:.4f}")
print(f"  Recall          : {metrics['recall_weighted']:.4f}")
print(f"  ECE (calibration): {metrics['ece']:.4f}")
if 'roc_auc_ovr' in metrics:
    print(f"  ROC-AUC (OvR)   : {metrics['roc_auc_ovr']:.4f}")
print(f"{'='*50}")

# per-class F1
print('\nPer-class F1:')
for cls, f1 in metrics['f1_per_class'].items():
    print(f'  {cls:<35} {f1:.4f}')

# save metrics JSON
import json
results_dir = Path(config['paths']['results_dir'])
results_dir.mkdir(parents=True, exist_ok=True)
with open(results_dir / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('\nMetrics saved to experiments/results/metrics.json')

In [ ]:
# ── 9. Grad-CAM visualisation ─────────────────────────────────────────
evaluator.visualise_gradcam(test_loader, n_samples=4)
print('Grad-CAM saved to', config['paths']['figures_dir'])

In [ ]:
# ── 10. ONNX export + INT8 quantization + latency benchmark ──────────
from src.evaluation.evaluator import export_and_quantize

onnx_result = export_and_quantize(trainer.model, config)
print(json.dumps(onnx_result, indent=2))

In [ ]:
# ── 11. Real-time inference demo ──────────────────────────────────────
from src.inference.realtime_detector import RealtimeDetector

detector = RealtimeDetector(
    config, audio_proc,
    model_path=str(Path(config['paths']['models_dir']) / 'best_model_overall.pth'),
    use_onnx=False)

# Run on a sample from test set
sample_row = test_df.iloc[0]
result = detector.predict_file(sample_row['local_path'])
print(f"\nReal-time prediction:")
print(f"  File      : {Path(sample_row['local_path']).name}")
print(f"  True label: {CLASS_NAMES[int(sample_row['label'])]}")
print(f"  Predicted : {result['species']}")
print(f"  Confidence: {result['confidence']:.3f}")
print(f"  Alert     : {result['alert']} ({result['priority']})")
print(f"  Latency   : {result['latency_ms']:.1f} ms")

In [ ]:
# ── 12. Project summary ───────────────────────────────────────────────
print('='*60)
print('BIRD CALL DETECTION — ML PROJECT SUMMARY')
print('='*60)
print(f"  Dataset        : {len(downloaded_df)} recordings, {NUM_CLASSES} classes")
print(f"  Accuracy       : {metrics['accuracy']:.4f}")
print(f"  F1 (weighted)  : {metrics['f1_weighted']:.4f}")
if USE_KFOLD:
    print(f"  K-Fold F1      : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  ONNX latency   : {onnx_result['latency_ms_fp32']:.1f} ms")
if 'latency_ms_int8' in onnx_result:
    print(f"  INT8 latency   : {onnx_result['latency_ms_int8']:.1f} ms")
print(f"  Figures        : {config['paths']['figures_dir']}")
print(f"  Models         : {config['paths']['models_dir']}")
print('='*60)